# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Árboles: no necesitan escalado, solo imputación por mediana
tree_preprocessor = ColumnTransformer(
    transformers=[("num", SimpleImputer(strategy="median"), feature_columns)],
    remainder="drop",
)

# Modelos sensibles a la escala (regresión logística y KNN): imputación + estandarización
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), feature_columns),
    ],
    remainder="drop",
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque el `StandardScaler` **aprende** estadísticos (media y desviación estándar) a partir de los datos. Si se ajusta con todo el conjunto antes de la validación cruzada, esos estadísticos ya incorporan información de las muestras que luego actúan como validación en cada fold: hay **fuga de datos** y la estimación de desempeño queda optimista. Dentro del `Pipeline`, `cross_validate` y `GridSearchCV` reajustan el escalador en cada fold usando solo sus muestras de entrenamiento y lo aplican (sin reajustar) a las de validación, que es exactamente lo que ocurrirá con datos nuevos en producción. Además, el pipeline guarda el escalador junto al modelo, así la inferencia aplica la misma transformación.

## 3. Modelos candidatos

In [5]:
models = {
    "decision_tree": Pipeline([
        ("preprocess", clone(tree_preprocessor)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("preprocess", clone(scaled_preprocessor)),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("preprocess", clone(scaled_preprocessor)),
        ("model", KNeighborsClassifier()),
    ]),
}
list(models)

['decision_tree', 'logistic_regression', 'knn']

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipe in models.items():
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv, scoring="accuracy", return_train_score=True,
    )
    rows.append({
        "model": name,
        "train_accuracy": scores["train_score"].mean(),
        "cv_accuracy_mean": scores["test_score"].mean(),
        "cv_accuracy_std": scores["test_score"].std(),
    })

initial_comparison = (
    pd.DataFrame(rows)
    .assign(gap=lambda d: d["train_accuracy"] - d["cv_accuracy_mean"])
    .sort_values("cv_accuracy_mean", ascending=False)
    .reset_index(drop=True)
)
initial_comparison.round(4)

,model,train_accuracy,cv_accuracy_mean,cv_accuracy_std,gap
0,logistic_regression,1.0000,0.9791,0.0277,0.0209
1,knn,0.9771,0.9507,0.0353,0.0264
2,decision_tree,1.0000,0.9022,0.0586,0.0978


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "decision_tree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 2, 5],
    },
    "logistic_regression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
    },
    "knn": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []
for name, pipe in models.items():
    search = GridSearchCV(
        estimator=clone(pipe),
        param_grid=param_grids[name],
        scoring="accuracy",
        cv=cv,
        refit=True,
        return_train_score=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search

    best = search.best_index_
    rows.append({
        "model": name,
        "train_accuracy": search.cv_results_["mean_train_score"][best],
        "cv_accuracy_mean": search.best_score_,
        "cv_accuracy_std": search.cv_results_["std_test_score"][best],
        "best_params": search.best_params_,
    })

tuned_comparison = (
    pd.DataFrame(rows)
    .assign(gap=lambda d: d["train_accuracy"] - d["cv_accuracy_mean"])
    .sort_values(["cv_accuracy_mean", "cv_accuracy_std"], ascending=[False, True])
    .reset_index(drop=True)
)
tuned_comparison.round(4)

,model,train_accuracy,cv_accuracy_mean,cv_accuracy_std,best_params,gap
0,logistic_regression,1.0000,0.9791,0.0277,{'model__C': 1},0.0209
1,knn,0.9648,0.9579,0.0405,"{'model__n_neighbors': 7, 'model__weights': 'u...",0.0069
2,decision_tree,0.9859,0.9025,0.0762,"{'model__max_depth': 5, 'model__min_samples_le...",0.0835


## 6. Selección y persistencia del campeón

In [8]:
# Campeón: mayor accuracy de CV; ante empate, menor variabilidad entre folds
champion_name = tuned_comparison.iloc[0]["model"]
champion = searches[champion_name].best_estimator_   # Pipeline completo, reajustado con todo train

print("Campeón:", champion_name)
print("Hiperparámetros:", searches[champion_name].best_params_)

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

import sklearn
training_metadata = {
    "champion": champion_name,
    "best_params": searches[champion_name].best_params_,
    "cv_accuracy_mean": float(tuned_comparison.iloc[0]["cv_accuracy_mean"]),
    "cv_accuracy_std": float(tuned_comparison.iloc[0]["cv_accuracy_std"]),
    "cv_folds": 5,
    "scoring": "accuracy",
    "random_state": RANDOM_STATE,
    "features": feature_columns,
    "n_train_rows": int(len(X_train)),
    "candidates": {
        r["model"]: {"cv_accuracy_mean": float(r["cv_accuracy_mean"]), "cv_accuracy_std": float(r["cv_accuracy_std"])}
        for _, r in tuned_comparison.iterrows()
    },
    "sklearn_version": sklearn.__version__,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
)

cv_results = pd.concat([
    initial_comparison.assign(stage="initial", best_params=""),
    tuned_comparison.assign(stage="tuned"),
], ignore_index=True)[["stage", "model", "train_accuracy", "cv_accuracy_mean", "cv_accuracy_std", "gap", "best_params"]]
cv_results["best_params"] = cv_results["best_params"].astype(str)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)
cv_results.round(4)

Campeón: logistic_regression
Hiperparámetros: {'model__C': 1}


,stage,model,train_accuracy,cv_accuracy_mean,cv_accuracy_std,gap,best_params
0,initial,logistic_regression,1.0000,0.9791,0.0277,0.0209,
1,initial,knn,0.9771,0.9507,0.0353,0.0264,
2,initial,decision_tree,1.0000,0.9022,0.0586,0.0978,
3,tuned,logistic_regression,1.0000,0.9791,0.0277,0.0209,{'model__C': 1}
4,tuned,knn,0.9648,0.9579,0.0405,0.0069,"{'model__n_neighbors': 7, 'model__weights': 'u..."
5,tuned,decision_tree,0.9859,0.9025,0.0762,0.0835,"{'model__max_depth': 5, 'model__min_samples_le..."


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

**Campeón: regresión logística** (`C=1`, preprocesamiento con mediana + `StandardScaler`).

| Modelo | Accuracy CV | Desv. CV | Accuracy train | Brecha train − CV |
|---|---|---|---|---|
| Regresión logística | **0.979** | **0.028** | 1.000 | 0.021 |
| KNN (k=7, uniforme) | 0.958 | 0.041 | 0.965 | 0.007 |
| Árbol (prof. 5, min_leaf 2) | 0.902 | 0.076 | 0.986 | 0.084 |

- **Desempeño:** la regresión logística tiene la mayor accuracy media en CV. La diferencia con KNN (~2 puntos) es menor que una desviación estándar entre folds, así que por sí sola no es concluyente; pero ningún otro criterio favorece a KNN.
- **Variabilidad:** también es el modelo más estable entre folds (0.028 frente a 0.041 y 0.076).
- **Sobreajuste:** el árbol es el que más se sobreajusta (0.986 en train frente a 0.902 en validación). KNN tiene la menor brecha, pero porque rinde menos en train y en validación. La regresión logística tiene una brecha pequeña (≈ 2 puntos).
- **Complejidad:** 3 × 13 coeficientes + 3 interceptos; entrenamiento e inferencia muy baratos. KNN debe guardar todo el set de entrenamiento y calcular distancias en cada predicción.
- **Interpretabilidad:** con variables estandarizadas, los coeficientes son comparables entre sí, aunque hay que leerlos con cuidado por la colinealidad vista en el EDA. El árbol es el más fácil de leer, pero es el menos preciso y el más inestable.

La selección se hizo solo con validación cruzada sobre train; `test.csv` no se cargó en este notebook.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.